In [17]:
import os
import fitz  # PyMuPDF
import pdfplumber
import uuid
from pathlib import Path
from tqdm import tqdm

from llama_index.core import Document, StorageContext
from llama_index.core.indices.knowledge_graph import KnowledgeGraphIndex
from llama_index.graph_stores.neo4j import Neo4jGraphStore
from llama_index.embeddings.openai import OpenAIEmbedding


def extract_structured_text(pdf_path):
    doc = fitz.open(pdf_path)

    blocks = []
    for page_num, page in enumerate(doc):
        text_blocks = page.get_text("dict")["blocks"]

        for block in text_blocks:
            if "lines" not in block:
                continue

            block_text = ""
            font_sizes = []

            for line in block["lines"]:
                for span in line["spans"]:
                    block_text += span["text"]
                    font_sizes.append(span["size"])

            if block_text.strip():
                avg_font = sum(font_sizes) / len(font_sizes)
                blocks.append({
                    "text": block_text.strip(),
                    "font_size": avg_font,
                    "page": page_num
                })

    return blocks


def detect_sections(blocks):
    if not blocks:
        return []

    max_font = max(b["font_size"] for b in blocks)

    structured = []
    current_section = "Unknown"
    current_content = []

    for block in blocks:
        # If font size close to max → treat as heading
        if block["font_size"] >= max_font * 0.9:
            if current_content:
                structured.append({
                    "section": current_section,
                    "content": "\n".join(current_content)
                })
                current_content = []

            current_section = block["text"]
        else:
            current_content.append(block["text"])

    if current_content:
        structured.append({
            "section": current_section,
            "content": "\n".join(current_content)
        })

    return structured

def detect_sections(blocks):
    if not blocks:
        return []

    max_font = max(b["font_size"] for b in blocks)

    structured = []
    current_section = "Unknown"
    current_content = []

    for block in blocks:
        if block["font_size"] >= max_font * 0.9:
            if current_content:
                structured.append({
                    "section": current_section,
                    "content": current_content  # KEEP AS LIST OF BLOCKS
                })
                current_content = []

            current_section = block["text"]
        else:
            current_content.append(block)  # keep full dict

    if current_content:
        structured.append({
            "section": current_section,
            "content": current_content
        })

    return structured

def recursive_chunk(text, max_tokens=800, overlap=100):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = min(start + max_tokens, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end == len(words):
            break

        start = max(end - overlap, 0)

    return chunks


def hybrid_chunking(structured_sections):
    documents = []

    for section in structured_sections:
        section_name = section["section"]
        blocks = section["content"]

        full_text = " ".join([b["text"] for b in blocks])
        pages = list(set([b["page"] for b in blocks]))

        sub_chunks = recursive_chunk(full_text)

        for i, chunk in enumerate(sub_chunks):
            documents.append(
                Document(
                    text=chunk,
                    metadata={
                        "doc_id": str(uuid.uuid4()),
                        "section": section_name,
                        "chunk_id": i,
                        "pages": pages
                    }
                )
            )

    return documents

In [18]:
def extract_tables(pdf_path):
    table_docs = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables = page.extract_tables()

            for i, table in enumerate(tables):
                table_text = "\n".join(
                    [" | ".join([str(cell) for cell in row]) for row in table]
                )

                table_docs.append(
                    Document(
                        text=table_text,
                        metadata={
                            "type": "table",
                            "page": page_num,
                            "table_index": i
                        }
                    )
                )

    return table_docs

def extract_images(pdf_path, image_dir="images"):
    Path(image_dir).mkdir(exist_ok=True)

    doc = fitz.open(pdf_path)
    image_docs = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        images = page.get_images(full=True)

        for img_index, img in enumerate(images):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            img_path = f"{image_dir}/{uuid.uuid4()}.png"

            with open(img_path, "wb") as f:
                f.write(image_bytes)

            image_docs.append(
                Document(
                    text=f"Image extracted from page {page_num}",
                    metadata={
                        "type": "image",
                        "page": page_num,
                        "image_path": img_path
                    }
                )
            )

    return image_docs

In [19]:
def process_pdf(pdf_path):
    blocks = extract_structured_text(pdf_path)
    structured = detect_sections(blocks)

    text_docs = hybrid_chunking(structured)
    table_docs = extract_tables(pdf_path)
    image_docs = extract_images(pdf_path)

    return text_docs + table_docs + image_docs



In [20]:
def ingest_to_neo4j(documents):

    graph_store = Neo4jGraphStore(
        username=os.getenv("NEO4J_USERNAME"),
        password=os.getenv("NEO4J_PASSWORD"),
        url=os.getenv("NEO4J_URI"),
        database=os.getenv("NEO4J_DATABASE"),
    )

    storage_context = StorageContext.from_defaults(graph_store=graph_store)

    index = KnowledgeGraphIndex.from_documents(
        documents,
        storage_context=storage_context,
        include_embeddings=True
    )
    return index




In [21]:
from pathlib import Path

pdf_folder = Path("research_pdfs")

print("Working directory:", Path().resolve())
print("Folder exists:", pdf_folder.exists())

pdf_files = list(pdf_folder.glob("*.pdf"))
print("PDF files found:", len(pdf_files))
print(pdf_files)

Working directory: C:\Users\Parth\Downloads\Agentic_Rag\research_paper
Folder exists: True
PDF files found: 88
[WindowsPath('research_pdfs/A defined N6methyladenosine m6A profile conferred by METT.pdf'), WindowsPath('research_pdfs/A positivefeedback loop between HBx and ALKBH5 pr.pdf'), WindowsPath('research_pdfs/Alkbh5 plays indispensable roles in maintaining se.pdf'), WindowsPath('research_pdfs/ALKBH5 promotes hypopharyngeal squamous cell carci.pdf'), WindowsPath('research_pdfs/ALKBH5 regulates IGF1R expression to promote the P.pdf'), WindowsPath('research_pdfs/ALKBH5 suppresses malignancy of hepatocellular car.pdf'), WindowsPath('research_pdfs/ALKBH5 suppresses tumor progression via an m6Adep.pdf'), WindowsPath('research_pdfs/Autophagy of the m6A mRNA demethylase FTO is impai.pdf'), WindowsPath('research_pdfs/Binding to m6A RNA promotes YTHDF2mediated phase.pdf'), WindowsPath('research_pdfs/C5aR1positive neutrophils promote breast cancer g.pdf'), WindowsPath('research_pdfs/CLK1SRSF5

In [22]:
from pathlib import Path
from tqdm import tqdm

# 1. Wrap your string in a Path object
pdf_folder = Path("research_pdfs/") 

all_documents = []

# 2. Now .glob() will work perfectly!
for pdf_file in tqdm(list(pdf_folder.glob("*.pdf"))):
    docs = process_pdf(pdf_file)
    all_documents.extend(docs)

 15%|█▍        | 13/88 [00:41<03:54,  3.12s/it]Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 

MuPDF error: format error: out of range code encountered in lzw decode

MuPDF error: library error: FT_New_Memory_Face(JMIMMU+HelveticaNeueLTStd-Lt): unknown file format



100%|██████████| 88/88 [05:00<00:00,  3.42s/it]
